In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import torch.optim as optim
import torch.nn as nn
import torch

import sentencepiece as spm

import math
import os
import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Hyper-parameters

In [ ]:
PAD_ID, SOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

# Model and dataset paths
SRC_FILE = "corpus/train.skz"
TGT_FILE = "corpus/train.zh"
VAL_SRC = "corpus/val.skz"
VAL_TGT = "corpus/val.zh"
SP_SRC_MODEL = "models/sp_merged_skz.model"
SP_TGT_MODEL = "models/sp_merged_zh.model"
SAVE_PATH = "models/model_best.pt"

# Dataloader settings (optimized for large corpus)
MAX_SEQ_LEN = 128
NUM_WORKERS = 4
PIN_MEMORY = True

# Model architecture (increased capacity for complex cipher mapping)
D_MODEL = 512
NHEAD = 8
NUM_LAYERS = 6
DIM_FEEDFORWARD = 2048
DROPOUT = 0.1
SP_MAX_LEN = 5000

# Training settings
BATCH_SIZE = 64
EPOCHS = 15
LR = 3e-4
WARMUP_STEPS = 500  # Reduced from 4000 to avoid gradient dormancy
BETAS = (0.9, 0.98)
EPS = 1e-9
GRAD_CLIP_NORM = 1.0
LABEL_SMOOTHING = 0.1

# Early stopping config
EARLY_STOP_PATIENCE = 2

## Load Tokenizer & Dataset

In [ ]:
src_sp = spm.SentencePieceProcessor()
src_sp.Load(model_file=SP_SRC_MODEL)
tgt_sp = spm.SentencePieceProcessor()
tgt_sp.Load(model_file=SP_TGT_MODEL)

class TranslationDataset(Dataset):
    def __init__(self, src_path, tgt_path, max_len=MAX_SEQ_LEN):
        with open(src_path, encoding="utf-8") as f:
            self.src_lines = f.readlines()
        with open(tgt_path, encoding="utf-8") as f:
            self.tgt_lines = f.readlines()
        # Strict alignment check
        assert len(self.src_lines) == len(self.tgt_lines), \
            f"Source and target line count mismatch: {len(self.src_lines)} vs {len(self.tgt_lines)}"
        self.max_len = max_len

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        s = [SOS_ID] + src_sp.encode(self.src_lines[idx].strip())[:self.max_len-2] + [EOS_ID]
        t = [SOS_ID] + tgt_sp.encode(self.tgt_lines[idx].strip())[:self.max_len-2] + [EOS_ID]
        return torch.tensor(s, dtype=torch.long), torch.tensor(t, dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_pad = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_ID)
    tgt_pad = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_ID)
    return src_pad, tgt_pad

# Load datasets
train_dataset = TranslationDataset(SRC_FILE, TGT_FILE)
val_dataset = TranslationDataset(VAL_SRC, VAL_TGT)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS // 2,
    pin_memory=PIN_MEMORY
)

print(f"Train: {len(train_dataset)} pairs | Val: {len(val_dataset)} pairs")

## Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=SP_MAX_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=D_MODEL, nhead=NHEAD, 
                 num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=DIM_FEEDFORWARD, 
            dropout=dropout, batch_first=True
        )
        self.out = nn.Linear(d_model, tgt_vocab)
        self.d_model = d_model

    def forward(self, src, tgt, src_pad_mask=None, tgt_pad_mask=None, tgt_mask=None):
        src = self.pos_enc(self.src_emb(src) * math.sqrt(self.d_model))
        tgt = self.pos_enc(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        memory = self.transformer.encoder(src, src_key_padding_mask=src_pad_mask)
        out = self.transformer.decoder(
            tgt, memory, 
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask, 
            memory_key_padding_mask=src_pad_mask
        )
        return self.out(out)

model = Seq2SeqTransformer(
    src_vocab=src_sp.vocab_size(),
    tgt_vocab=tgt_sp.vocab_size(),
    d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS, dropout=DROPOUT
)
model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Inference

In [ ]:
def beam_search(model, src_text, src_sp, tgt_sp, beam_size=5, max_len=64, length_penalty=0.6):
    model.eval()
    device = next(model.parameters()).device
    
    # Encode source
    src_ids = torch.tensor([SOS_ID] + src_sp.encode(src_text) + [EOS_ID], dtype=torch.long).unsqueeze(0).to(device)
    src_pad_mask = (src_ids == PAD_ID)
    
    # Initialize beam
    beams = [{"seq": [SOS_ID], "score": 0.0, "finished": False}]
    
    for step in range(1, max_len):
        active = [b for b in beams if not b["finished"]]
        if not active:
            break
            
        # Batchify target sequences
        batch_seqs = [torch.tensor(b["seq"], dtype=torch.long, device=device).unsqueeze(0) for b in active]
        seqs_tensor = torch.cat(batch_seqs, dim=0)
        tgt_pad_mask = (seqs_tensor == PAD_ID)
        sz = seqs_tensor.size(1)
        tgt_mask = generate_bool_causal_mask(sz, device)
        
        # Expand source to match batch size
        src_exp = src_ids.expand(seqs_tensor.size(0), -1)
        src_pad_exp = src_pad_mask.expand(seqs_tensor.size(0), -1)
        
        with torch.no_grad():
            out = model(src_exp, seqs_tensor, tgt_mask=tgt_mask, 
                        src_pad_mask=src_pad_exp, tgt_pad_mask=tgt_pad_mask)
            
        log_probs = torch.log_softmax(out[:, -1, :], dim=-1)
        
        # Collect candidates
        candidates = []
        for i, b in enumerate(active):
            for token_id in range(log_probs.size(1)):
                prob = log_probs[i, token_id].item()
                new_score = b["score"] + prob
                
                new_seq = b["seq"] + [token_id]
                finished = token_id == EOS_ID
                
                # Apply length penalty ONLY to finished sequences or max-length sequences
                if finished or step == max_len - 1:
                    new_score /= ((5 + len(new_seq)) / 6) ** length_penalty
                    
                candidates.append({"seq": new_seq, "score": new_score, "finished": finished})
                
        # Keep top-k
        candidates.sort(key=lambda x: x["score"], reverse=True)
        beams = candidates[:beam_size]
        
    # Return best sequence (remove SOS)
    best_seq = beams[0]["seq"][1:]
    if best_seq and best_seq[-1] == EOS_ID:
        best_seq = best_seq[:-1]
        
    return tgt_sp.decode(best_seq)

## Train
### Training components with stability fixes

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, betas=BETAS, eps=EPS, weight_decay=1e-4)

def lr_schedule(step):
    step += 1
    return min(step * WARMUP_STEPS**-1.5, step**-0.5)

scheduler = LambdaLR(optimizer, lr_lambda=lr_schedule)

# Explicit bool mask constructor to eliminate PyTorch warning
def generate_bool_causal_mask(sz, device):
    return ~torch.tril(torch.ones(sz, sz, device=device, dtype=torch.bool))

### Validation function

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in = tgt[:, :-1]
            sz = tgt_in.size(1)
            tgt_mask = generate_bool_causal_mask(sz, device)
            src_pad = src == PAD_ID
            tgt_pad = tgt_in == PAD_ID
            
            out = model(src, tgt_in, src_pad_mask=src_pad, tgt_pad_mask=tgt_pad, tgt_mask=tgt_mask)
            loss = criterion(out.transpose(1, 2), tgt[:, 1:])
            total_loss += loss.item()
    model.train()
    return total_loss / len(loader)

### Main loop

In [ ]:
print("Starting training...")
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    # Training phase
    model.train()
    total_train_loss = 0
    
    for src, tgt in tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        src, tgt = src.to(device), tgt.to(device)
        tgt_in = tgt[:, :-1]
        sz = tgt_in.size(1)
        
        # Use explicit bool mask to avoid type mismatch warning
        tgt_mask = generate_bool_causal_mask(sz, device)
        src_pad = src == PAD_ID
        tgt_pad = tgt_in == PAD_ID

        optimizer.zero_grad()
        out = model(src, tgt_in, src_pad_mask=src_pad, tgt_pad_mask=tgt_pad, tgt_mask=tgt_mask)
        loss = criterion(out.transpose(1, 2), tgt[:, 1:])
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation phase
    avg_val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
    
    # Checkpoint strategy: save only best model by validation loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
        }, SAVE_PATH)
        print(f"  -> Saved new best model to {SAVE_PATH}")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> Val loss did not improve ({patience_counter}/{EARLY_STOP_PATIENCE})")
    
    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered after {epoch+1} epochs")
        break

print(f"Training complete. Best validation loss: {best_val_loss:.4f}")

NameError: name 'optimizer' is not defined

### Test

In [ ]:
# Quick test with best checkpoint
if os.path.exists(SAVE_PATH):
    checkpoint = torch.load(SAVE_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model (epoch {checkpoint['epoch']+1}, val_loss {checkpoint['val_loss']:.4f})")
    
    test_cipher = "ytqqrvqbsbtjyrernx"
    result = beam_search(model, test_cipher, src_sp, tgt_sp, beam_size=5, max_len=64)
    print(f"Test input: {test_cipher}")
    print(f"Test output: {result}")

/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


是一位意大利的游戏。 作品
NGC 2 2 2 2 2
